# Ver los resultados de `busqueda.py`

Corré las celdas en orden (o *Run All*). Este notebook **solo lee** la carpeta `resultados/`,
así que lo podés volver a correr mientras la búsqueda sigue andando para ver cómo va.

Cómo leer los números:
- **μ** = grado mínimo de la plantilla: cada oscilador está conectado a esa fracción de los demás.
  Es lo que queremos que sea lo más grande posible.
- **11/16 = 0.6875** es la cota de Canale (2022); **0.6929077** es el récord actual (O'Keeffe, oct. 2026).
  Para mejorar la cota hay que pasar **0.6929077**.
- **λ2** = margen de estabilidad (tiene que ser > 0). Todo esto es en punto flotante: un μ que supera el récord
  es un *candidato*, no una prueba, hasta que se certifique.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plantillas as T

CARPETA = "resultados"
CANALE, RECORD = 11 / 16, 0.6929077

# encoding_errors="replace": en Windows las versiones anteriores escribían el csv en cp1252
df = pd.read_csv(os.path.join(CARPETA, "resultados.csv"), encoding="utf-8", encoding_errors="replace")
df = df[df.mu > 0].sort_values("mu", ascending=False).reset_index(drop=True)
# familia del punto de partida (s7, c4_partido, canale, twisted); las que "continúan"
# una búsqueda anterior heredan la familia de esa búsqueda
df["tipo"] = df.familia if "familia" in df else df.inicio.str.extract(r"^(canale|s7|twisted|continúa)")[0]

print(f"Búsquedas terminadas: {len(df)}")
print(f"Superan 11/16:        {(df.mu > CANALE).sum()}")
print(f"Superan el récord:    {(df.mu > RECORD).sum()}")
print(f"Mejor μ:              {df.mu.max():.7f}   (le falta {RECORD - df.mu.max():.2e} para el récord)")
print(f"Búsquedas que llegaron exactamente al mejor μ: {(df.mu > df.mu.max() - 1e-8).sum()}")

# Ranking de plantillas DISTINTAS: muchas búsquedas que continúan el mismo punto terminan
# en el mismo valor; nos quedamos con una por valor de μ y contamos cuántas lo repiten.
df["valor"] = df.mu.round(8)
distintas = (df.groupby("valor").agg(id=("id", "first"), mu=("mu", "first"), repeticiones=("id", "size"),
                                     familia=("tipo", "first"), m_final=("m_final", "first"),
                                     lambda2=("lambda2", "first"))
             .sort_values("mu", ascending=False).reset_index(drop=True))

## 1. Ranking de plantillas distintas

Las 10 mejores plantillas **distintas**. Muchas búsquedas que siguen desde el mismo punto terminan en exactamente
el mismo μ; acá aparece una sola vez cada valor, y `repeticiones` dice cuántas búsquedas llegaron a él.
`id` es la primera búsqueda que lo encontró (es el que hay que mirar en la sección 4).

In [ ]:
distintas.head(10).style.format({"mu": "{:.9f}", "lambda2": "{:.1e}"})

## 2. ¿Qué tan bien le va a cada punto de partida?

Cada punto es una búsqueda, coloreada por su familia (desde qué plantilla conocida arrancó; las que
continúan otra búsqueda heredan su familia). Las líneas horizontales son Canale (gris) y el récord (rojo).
Si todos los puntos de un color quedan en el mismo valor, ese punto de partida está "atascado" ahí.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for tipo, g in df.groupby("tipo"):
    ax.scatter(g.index, g.mu, label=tipo, s=18)
ax.axhline(CANALE, color="gray", ls="--", lw=1, label="Canale 11/16")
ax.axhline(RECORD, color="red", ls="--", lw=1, label="récord O'Keeffe")
ax.set_ylim(max(0.65, df.mu.min() - 0.002), max(RECORD, df.mu.max()) + 0.001)
ax.set_xlabel("búsquedas (ordenadas de mejor a peor)"); ax.set_ylabel("μ")
ax.legend(fontsize=8); plt.show()

print(df.groupby("tipo").mu.agg(["count", "max", "median"]).sort_values("max", ascending=False))

## 3. Cómo avanzó cada búsqueda

Mejor μ en función de la iteración, para las 5 mejores. Si la curva sigue subiendo al final, vale la pena
darle más iteraciones (`--iters`). Si se aplanó enseguida, la búsqueda se estancó.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for i in range(min(5, len(df))):
    h = np.load(os.path.join(CARPETA, df.id[i] + ".npz"))["historia"]
    ax.plot(np.where(h > 0, h, np.nan), label=df.id[i])
ax.axhline(RECORD, color="red", ls="--", lw=1)
ax.set_xlabel("iteración"); ax.set_ylabel("mejor μ hasta ese momento")
ax.legend(fontsize=8); plt.show()

## 4. Mirar una plantilla en detalle

Por defecto muestra la mejor. Para ver otra, cambiá `ID` por cualquier id del ranking.

In [ ]:
ID = df.id[0]          # <- cambiá esto para ver otra plantilla

d = np.load(os.path.join(CARPETA, ID + ".npz"))
x, phi, P = d["x"], d["phi"], d["P"]
orden = np.argsort(phi)                       # ordenamos los grupos por fase
x, phi, P = x[orden], phi[orden], P[np.ix_(orden, orden)]

print(f"Plantilla {ID}: μ = {float(d['mu']):.7f}, {len(x)} grupos, arrancó desde {d['inicio']}\n")
print(pd.DataFrame({"tamaño x_a": x.round(4), "fase (grados)": np.degrees(phi).round(4),
                    "grado d_a": (P @ x).round(6)}))
print("\nChequeo:", T.verificar(x, phi, P))

### Matriz de densidades P

`P[a, b]` = fracción de las aristas posibles entre los grupos a y b que existen (1 = todas, 0 = ninguna).
Los valores intermedios son los "raleos" que hacen estable al estado (en S7 son ≈ 0.65).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(P, vmin=0, vmax=1, cmap="viridis")
for a in range(len(x)):
    for b in range(len(x)):
        ax.text(b, a, f"{P[a, b]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if P[a, b] < 0.5 else "black")
etiquetas = [f"{np.degrees(p):.1f}°" for p in phi]
ax.set_xticks(range(len(x)), etiquetas, rotation=90, fontsize=7)
ax.set_yticks(range(len(x)), etiquetas, fontsize=7)
plt.colorbar(im, label="densidad"); plt.title("P (grupos ordenados por fase)"); plt.show()

### La plantilla dibujada en el círculo de fases

Cada punto es un grupo, en su fase; el área es proporcional a su tamaño. Cada cuerda es una conexión entre grupos:
gris = completa, naranja = raleada (densidad intermedia). Las marcas son los cuartos de vuelta.

Los grupos partidos están separados por ángulos diminutos (~0.1°), así que se **exagera** la separación
respecto del cuarto de vuelta más cercano (`EXAGERAR`), como en la Fig. 1 del paper.

In [ ]:
EXAGERAR = 40          # 1 = ángulos reales

cuarto = np.round(phi / (np.pi / 2)) * (np.pi / 2)
ang = cuarto + EXAGERAR * (phi - cuarto)
px, py = np.cos(ang), np.sin(ang)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="lightgray"))
for t in np.arange(4) * np.pi / 2:
    ax.plot([0.95 * np.cos(t), 1.05 * np.cos(t)], [0.95 * np.sin(t), 1.05 * np.sin(t)], color="gray")
for a in range(len(x)):
    for b in range(a + 1, len(x)):
        if P[a, b] > 1e-3:
            color = "gray" if P[a, b] > 0.98 else "darkorange"
            ax.plot([px[a], px[b]], [py[a], py[b]], color=color,
                    lw=60 * x[a] * x[b] * P[a, b], alpha=0.7, zorder=1)
ax.scatter(px, py, s=3000 * x, color="tab:blue", zorder=2)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"{ID}: μ = {float(d['mu']):.6f}  (separaciones ×{EXAGERAR})")
plt.show()

## 5. Las mejores de cada familia

Un gráfico por familia (desde qué plantilla conocida arrancó: S7, C4 partido, Canale, twisted) con sus `TOP`
mejores plantillas **distintas**. Cada punto es una plantilla; a la izquierda, su μ y el id de la primera búsqueda
que lo encontró. Las líneas punteadas son Canale (11/16) y el récord, cuando caen dentro del rango del panel.
Cada panel tiene su propia escala, para que se vean las diferencias dentro de la familia.

In [ ]:
TOP = 5          # <- cuántas plantillas distintas mostrar por familia

ORDEN = ["s7", "c4_partido", "canale", "twisted"]
familias = [f for f in ORDEN if f in set(df.tipo)] + sorted(set(df.tipo.dropna()) - set(ORDEN))

# una fila por plantilla distinta: si varias búsquedas llegaron al mismo μ, queda la primera (id más viejo)
mejores = {f: (df[df.tipo == f].sort_values(["mu", "id"], ascending=[False, True])
               .drop_duplicates("valor").head(TOP).reset_index(drop=True))
           for f in familias}

AZUL, GRIS, TINTA = "#2a78d6", "#52514e", "#0b0b0b"
filas = (len(familias) + 1) // 2
fig, axs = plt.subplots(filas, 2, figsize=(12, 2.9 * filas), squeeze=False)
for ax, f in zip(axs.flat, familias):
    g = mejores[f]
    y = np.arange(len(g))[::-1]                       # la mejor arriba
    lo, hi = g.mu.min(), g.mu.max()
    pad = max((hi - lo) * 0.15, 2e-5)
    ax.hlines(y, lo - pad, g.mu, color=AZUL, alpha=0.25, lw=2)
    ax.scatter(g.mu, y, s=64, color=AZUL, zorder=3)
    for ref, nombre in [(CANALE, "Canale 11/16"), (RECORD, "récord")]:
        if lo - pad <= ref <= hi + 3 * pad:
            ax.axvline(ref, color=GRIS if ref == CANALE else TINTA, ls="--", lw=1)
            ax.text(ref, len(g) - 0.4, f" {nombre}", fontsize=8, color=GRIS, va="bottom")
    ax.set_xlim(lo - pad, max([hi + pad] + [r + pad for r in (CANALE, RECORD) if lo - pad <= r <= hi + 3 * pad]))
    ax.set_ylim(-0.6, len(g) - 0.2)
    ax.set_yticks(y, [f"{m:.7f}  ·  {i}" for m, i in zip(g.mu, g.id)], fontsize=8.5)
    ax.set_title(f"{f}: {int((df.tipo == f).sum())} búsquedas, mejor μ = {hi:.7f} "
                 f"(le falta {RECORD - hi:.1e} al récord)", fontsize=10, loc="left", color=TINTA)
    ax.xaxis.set_major_locator(plt.MaxNLocator(5))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.6f}"))
    ax.tick_params(axis="x", labelsize=8, colors=GRIS)
    ax.grid(axis="x", color="#e4e3df", lw=0.8)
    ax.set_axisbelow(True)
    for lado in ["top", "right", "left"]:
        ax.spines[lado].set_visible(False)
    ax.set_xlabel("μ", fontsize=9, color=GRIS)
for ax in list(axs.flat)[len(familias):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Condiciones para replicarlas

Para cada una de esas plantillas, la celda de abajo imprime dos cosas:

- **La plantilla misma (x, φ):** es lo que define el grafo. Copiando esas dos listas y corriendo
  `mu, P = T.mejor_densidad(x, phi, eps)` se recupera exactamente el mismo μ y la matriz de densidades P.
  Es la forma segura de replicar el resultado.
- **Cómo se llegó a ella:** el punto de partida, la semilla y los parámetros de la búsqueda, y la cadena de
  búsquedas si "continúa" otra. Las búsquedas lanzadas con la versión nueva de `busqueda.py` guardan además el
  punto de partida exacto (x₀, φ₀), así que se pueden repetir con `T.buscar`. Las de versiones anteriores no lo
  guardaban: para esas, solo la plantilla final es reproducible.

`SOLO_LA_MEJOR = True` imprime solo la mejor de cada familia; con `False` imprime las `TOP`.

In [ ]:
SOLO_LA_MEJOR = True

todas = pd.read_csv(os.path.join(CARPETA, "resultados.csv"), encoding="utf-8", encoding_errors="replace")
por_id = todas.set_index("id")

def cadena(i):
    # sigue los "continúa <id>" hacia atrás hasta el punto de partida original
    pasos = [i]
    while True:
        ini = str(por_id.loc[pasos[-1], "inicio"]) if pasos[-1] in por_id.index else ""
        padre = pd.Series([ini]).str.extract(r"(\d{4}_\d{6}_\d+)")[0].iloc[0]
        if ini.startswith("contin") and isinstance(padre, str) and padre in por_id.index and padre not in pasos:
            pasos.append(padre)
        else:
            return pasos[::-1], ini

def lista(v):
    return "[" + ", ".join(repr(float(t)) for t in v) + "]"

for f in familias:
    g = mejores[f].head(1) if SOLO_LA_MEJOR else mejores[f]
    for k, r in g.iterrows():
        print("=" * 100)
        print(f"Familia {f} · #{k + 1} · {r.id} · μ = {r.mu:.10f} · {int(r.m_final)} grupos")
        pasos, raiz = cadena(r.id)
        print(f"  Punto de partida original: {raiz}")
        if len(pasos) > 1:
            print(f"  Cadena de búsquedas: {' -> '.join(pasos)}")
        print(f"  Parámetros de esta búsqueda: semilla={r.semilla}, eps={r.eps:g}, iters={int(r.iters)}, "
              f"temp={r.temp:g}, m_max={int(r.m_max)}")
        d = np.load(os.path.join(CARPETA, r.id + ".npz"))
        o = np.argsort(d["phi"])
        xb, phib = d["x"][o], d["phi"][o]
        print("\n  Plantilla final (grupos ordenados por fase):")
        print(f"    x   = {lista(xb)}")
        print(f"    phi = {lista(phib)}   # radianes")
        print(f"    fases en grados: {np.round(np.degrees(phib), 6).tolist()}")
        print(f"    -> mu, P = T.mejor_densidad(x, phi, eps={r.eps:g})")
        if "x0" in d.files:
            print("\n  Punto de partida exacto de esta búsqueda (para repetirla):")
            print(f"    x0   = {lista(d['x0'])}")
            print(f"    phi0 = {lista(d['phi0'])}")
            print(f"    -> T.buscar(x0, phi0, eps={r.eps:g}, iters={int(r.iters)}, m_max={int(r.m_max)}, "
                  f"semilla={r.semilla}, temp={r.temp:g}, escala_min={int(d['escala_min'])})")
        else:
            print("\n  (Esta búsqueda es de una versión anterior: no guardó su punto de partida exacto.)")
        mu_chk, P_chk = T.mejor_densidad(xb, phib, float(r.eps))
        if mu_chk is None:
            print("\n  Chequeo: con estos x, phi no se encontró plantilla estricta (revisar)")
            continue
        print(f"\n  Chequeo: recalculado μ = {mu_chk:.10f}  " + ("✓" if abs(mu_chk - r.mu) < 1e-8 else "≠ (revisar)"))
        print("  Densidades P:")
        print("    " + np.array2string(P_chk, precision=4, suppress_small=True, max_line_width=140, prefix="    "))

## 6. Si algo supera el récord

Antes de festejar:

1. Mirá en el chequeo de arriba que `max|F|` sea ~1e-15 (equilibrio exacto) y `lambda2 > 0`.
2. Corré la celda de abajo: recalcula el μ con márgenes distintos. Si μ sigue arriba de 0.6929 con `eps` = 1e-6,
   el candidato es robusto; si solo aparece con márgenes diminutos, puede ser ruido numérico.
3. Guardá el `.npz` y avisá: el paso siguiente es la certificación rigurosa (aritmética de intervalos).

In [ ]:
for eps in [1e-6, 1e-7, 1e-8, 1e-9]:
    mu_e, P_e = T.mejor_densidad(x, phi, eps)
    print(f"eps = {eps:.0e}:  " + ("sin plantilla estricta" if mu_e is None else
          f"μ = {mu_e:.7f}  {'> récord' if mu_e > RECORD else ''}"))